# 11. Interpretabilidad de Modelos de Machine Learning y Deep Learning

Este notebook explora técnicas modernas de interpretabilidad de modelos, incluyendo SHAP, LIME e importancia de características.

## Objetivo
- Comprender la importancia de la interpretabilidad en ML.
- Implementar y visualizar SHAP, LIME e importancia de características.
- Aprender a comunicar resultados de interpretabilidad.

## Prerequisitos

> 📌 **Prerequisitos:** Haber completado los notebooks [01](./01_intro_machine_learning.ipynb) y [03](./03_modelos_clasicos_ml.ipynb).

- Conceptos de modelos de clasificación, Random Forest, y métricas.

> ⚠️ **Dependencias adicionales:** `pip install shap lime`

## 1. Introducción teórica

La interpretabilidad permite entender **por qué** un modelo toma ciertas decisiones. Esto es esencial para:
- Validar y confiar en modelos antes de desplegarlos.
- Detectar sesgos y problemas en los datos.
- Cumplir con regulaciones (GDPR, IA responsable).

| Técnica | Alcance | Ventaja | Desventaja |
|---------|---------|---------|------------|
| **Feature Importance** | Global | Rápido, integrado en el modelo | Solo modelos de árboles |
| **SHAP** | Global + Local | Robusto, basado en teoría de juegos | Más costoso computacionalmente |
| **LIME** | Local | Rápido, agnóstico al modelo | Aproximación local, puede variar |

## 2. Importación de librerías

In [ ]:
# === Reproducibilidad ===
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap
import lime
import lime.lime_tabular

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 3. Entrenar un modelo de ejemplo

In [ ]:
data = load_breast_cancer()
features = data.feature_names
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

model = RandomForestClassifier(n_estimators=100, random_state=SEED)
model.fit(X_train, y_train)
print(f"Accuracy en test: {model.score(X_test, y_test):.2f}")

## 4. Importancia de características

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10,6))
plt.bar(range(len(importances)), importances[indices], color='steelblue')
plt.xticks(range(len(importances)), features[indices], rotation=90)
plt.title('Importancia de características (Random Forest)')
plt.tight_layout()
plt.show()

# Top 5
print('Top 5 variables más importantes:')
for i in range(5):
    print(f'  {i+1}. {features[indices[i]]}: {importances[indices[i]]:.4f}')

## 5. Explicaciones con SHAP

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot (importancia global con dirección del efecto)
shap.summary_plot(shap_values[1], X_test, feature_names=features, show=False)
plt.title('SHAP Summary Plot (clase benigno)')
plt.tight_layout()
plt.show()

In [ ]:
# Explicación local: primera muestra de test
shap.initjs()
shap.force_plot(explainer.expected_value[1], shap_values[1][0], X_test[0],
                feature_names=features, matplotlib=True)
plt.title('SHAP Force Plot - Muestra 0')
plt.show()

## 6. Explicaciones con LIME

In [ ]:
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train, feature_names=features,
    class_names=['Maligno', 'Benigno'],
    discretize_continuous=True)

# Explicar la primera predicción del test
exp = explainer_lime.explain_instance(X_test[0], model.predict_proba, num_features=10)
exp.show_in_notebook(show_table=True, show_all=False)

## 7. Discusión y Conclusiones

- La interpretabilidad es esencial para validar y confiar en modelos.
- **Feature Importance** es rápido pero limitado a modelos de árboles.
- **SHAP** es el más robusto: muestra importancia global y explicaciones locales.
- **LIME** es rápido y agnóstico al modelo, ideal para explicaciones locales.
- Documenta y comunica los resultados de interpretabilidad a stakeholders.

## 8. Ejercicios Propuestos

1. **Ejercicio 1:** Aplica SHAP a un modelo diferente (GradientBoosting, XGBoost). ¿Cambian las variables importantes?

2. **Ejercicio 2:** Compara las explicaciones LIME para muestras correctamente e incorrectamente clasificadas.

3. **Ejercicio 3:** Usa `shap.dependence_plot` para explorar la relación entre una variable y su efecto en la predicción.

4. **Ejercicio 4 (Avanzado):** Aplica SHAP a una red neuronal de Keras usando `shap.DeepExplainer` o `shap.KernelExplainer`.

## 9. Referencias y Recursos

- [Interpretable ML Book](https://christophm.github.io/interpretable-ml-book/)
- [SHAP Documentation](https://shap.readthedocs.io/)
- [LIME GitHub](https://github.com/marcotcr/lime)
- Lundberg & Lee (2017). *A Unified Approach to Interpreting Model Predictions.*

---

📎 **Notebook anterior:** [10. Clustering y Reducción](./10_clustering_reduccion_dimensionalidad.ipynb)  
📎 **Notebook siguiente:** [12. CPU, GPU y Metal](./12_cpu_gpu_metal.ipynb)